In [1]:
!uv pip install unsloth

Using Python 3.12.12 environment at: /usr
Resolved 102 packages in 1.01s
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
⠙ Preparing packages... (0/14)
dill                 ------------------------------ 78.88 KiB/116.86 KiB
⠙ Preparing packages... (0/14)
dill                 ------------------------------ 78.88 KiB/116.86 KiB
⠙ Preparing packages... (0/14)
dill                 ------------------------------ 94.88 KiB/116.86 KiB
⠙ Preparing packages... (0/14)
dill                 ------------------------------ 94.88 KiB/116.86 KiB
⠙ Preparing packages... (0/14)
dill                 ------------------------------ 94.88 KiB/116.86 KiB
⠙ Preparing packages... (0/14)
dill                 ------------------------------ 94.88 KiB/116.86 KiB
⠙ Preparing packages... (0/14)
dill                 ------------------------------ 94.

In [2]:
from unsloth import FastVisionModel
import torch
from datasets import load_dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
model, tokenizer = FastVisionModel.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct",
    use_gradient_checkpointing = "unsloth",
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.4.2: Fast Qwen3_Vl patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.41G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [4]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 64,
    lora_alpha = 64,
    lora_dropout = 0.1,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


In [5]:
dataset = load_dataset("HuggingFaceM4/ChartQA")

train_data = dataset["train"].shuffle(seed=42).select(range(200))
val_data = dataset["val"].shuffle(seed=42).select(range(20))
test_data = dataset["test"].shuffle(seed=42).select(range(20))

README.md:   0%|          | 0.00/852 [00:00<?, ?B/s]

data/train-00000-of-00003-49492f364babfa(…):   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00001-of-00003-7302bae5e425bb(…):   0%|          | 0.00/311M [00:00<?, ?B/s]

data/train-00002-of-00003-194c9400785577(…):   0%|          | 0.00/315M [00:00<?, ?B/s]

data/val-00000-of-00001-0f11003c77497969(…):   0%|          | 0.00/50.2M [00:00<?, ?B/s]

data/test-00000-of-00001-e2cd0b7a0f9eb20(…):   0%|          | 0.00/68.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/28299 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1920 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [6]:
dataset = load_dataset("HuggingFaceM4/ChartQA")

train_data = dataset["train"]
val_data = dataset["val"]
test_data = dataset["test"]

In [7]:
train_data

Dataset({
    features: ['image', 'query', 'label', 'human_or_machine'],
    num_rows: 28299
})

In [ ]:
system_message = """You are an expert chart and data visualization analyst.
When given a chart image and a question, analyze the visual data carefully and provide
a precise, concise answer. For numerical answers, provide the exact value shown in the chart.
For categorical answers, use the exact labels from the chart."""

def format_data(sample):
    label = sample["label"][0] if isinstance(sample["label"], list) else str(sample["label"])

    return {
        "messages": [                         
            {
                "role": "system",
                "content": [{"type": "text", "text": system_message}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": sample["image"]},
                    {"type": "text",  "text": sample["query"]},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": label}],
            },
        ]
    }



train_dataset = train_data.map(format_data, remove_columns=train_data.column_names)
val_dataset   = val_data.map(format_data,   remove_columns=val_data.column_names)
test_dataset  = test_data.map(format_data,  remove_columns=test_data.column_names)


Map:   0%|          | 0/28299 [00:00<?, ? examples/s]

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [9]:
# FastVisionModel.for_inference(model) # Enable for inference

# import PIL.Image
# import io

# sample_data_index = 2
# sample_data = train_dataset[sample_data_index]

# # Extract image bytes and convert to PIL Image
# image_bytes = sample_data['messages'][1]["content"][0]["image"]['bytes']
# image = PIL.Image.open(io.BytesIO(image_bytes))

# instruction = sample_data['messages'][1]["content"][1]["text"]

# messages = [
#     {"role": "user", "content": [
#         {"type": "image"},
#         {"type": "text", "text": instruction}
#     ]}
# ]
# input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
# inputs = tokenizer(
#     image,
#     input_text,
#     add_special_tokens = False,
#     return_tensors = "pt",
# ).to("cuda")

# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer, skip_prompt = True)
# _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
#                    use_cache = True, temperature = 1.5, min_p = 0.1)


In [10]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)


Unsloth: Model does not have a default image size - using 512


In [11]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28,299 | Num Epochs = 1 | Total steps = 885
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 94,896,128 of 2,222,428,160 (4.27% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.593713
2,2.608916
3,2.223238
4,1.579465
5,1.187947
6,0.836923
7,0.551604
8,0.443707
9,0.330860
10,0.402118


In [12]:
# if False:
#     from unsloth import FastVisionModel
#     model, tokenizer = FastVisionModel.from_pretrained(
#         model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
#         load_in_4bit = True, # Set to False for 16bit LoRA
#     )
#     FastVisionModel.for_inference(model) # Enable for inference!

# import PIL.Image
# import io

# # Access an image from the 'test' split of the dataset
# sample_data = test_data[10]

# # sample_data['image'] is already a PIL.Image object, so we use it directly
# image = sample_data['image']

# instruction = sample_data["query"]

# messages = [
#     {"role": "user", "content": [
#         {"type": "image"},
#         {"type": "text", "text": instruction}
#     ]}
# ]
# input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
# inputs = tokenizer(
#     image,
#     input_text,
#     add_special_tokens = False,
#     return_tensors = "pt",
# ).to("cuda")

# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer, skip_prompt = True)
# _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048,
#                    use_cache = True, temperature = 1.5, min_p = 0.1)

In [ ]:
from huggingface_hub import login
login(token = "")

In [ ]:
model.push_to_hub_merged(
    "shubhamprakash108/chartqa-vlm-vllm",
    tokenizer=tokenizer,
    save_method="merged_16bit",
    token=""
)

config.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:16<00:00, 76.96s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/shubhamprakash108/chartqa-vlm-vllm`
